### NOTE:- First disable the deletion vectors and perform operations. Later enable the deletion vectors and perform operations to see the difference.

### check deletion vectors enabled or not on path

In [0]:
spark.sql("""
SHOW TBLPROPERTIES delta.`/Volumes/tri/tri/tri_files/OUTPUT/products_delta`
('delta.enableDeletionVectors')
""").show(truncate=False)

### disable the deletion vector on path level

In [0]:
spark.sql("""
ALTER TABLE delta.`/Volumes/tri/tri/tri_files/OUTPUT/products_delta`
SET TBLPROPERTIES (
  'delta.enableDeletionVectors'='false'
)
""")

### Enable Deletion Vectors During Table Creation

In [0]:

CREATE TABLE employee (
    id INT,
    name STRING,
    salary DOUBLE
)
TBLPROPERTIES (
    'delta.enableDeletionVectors' = 'true'
);


### create dataframe

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, FloatType, IntegerType
# Define schema
schema = StructType([
    StructField("ProductID", StringType(), True),
    StructField("ProductName", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("Price", FloatType(), True),
    StructField("StockQuantity", IntegerType(), True)
])
data = [
    ("P001", "Laptop", "Electronics", 999.99, 50),
    ("P002", "Smartphone", "Electronics", 699.99, 150),
    ("P003", "Tablet", "Electronics", 299.99, 100),
    ("P004", "Smartwatch", "Electronics", 199.99, 75),
    ("P005", "Headphones", "Accessories", 149.99, 200),
    ("P006", "Camera", "Electronics", 499.99, 60),
    ("P007", "Printer", "Electronics", 299.99, 40),
    ("P008", "Monitor", "Electronics", 199.99, 80),
    ("P009", "Keyboard", "Accessories", 49.99, 150),
    ("P010", "Mouse", "Accessories", 29.99, 200),
    ("P011", "Router", "Networking", 79.99, 90),
    ("P012", "Speaker", "Audio", 99.99, 110),
    ("P013", "Webcam", "Accessories", 59.99, 130),
    ("P014", "Charger", "Accessories", 19.99, 170),
    ("P015", "Power Bank", "Accessories", 39.99, 120),
    ("P016", "USB Drive", "Storage", 24.99, 250),
    ("P017", "External HDD", "Storage", 79.99, 180),
    ("P018", "Memory Card", "Storage", 19.99, 140),
    ("P019", "Graphics Card", "Components", 249.99, 30),
    ("P020", "Microphone", "Audio", 59.99, 95)
]
# Create DataFrame
df = spark.createDataFrame(data, schema)
df.display()


In [0]:
# Write in Delta format
df.repartition(1).write \
    .format("delta") \
    .mode("append") \
    .save("/Volumes/tri/tri/tri_files/OUTPUT/products_delta")


In [0]:
spark.sql("""
CREATE TABLE product_catalog
USING DELTA
LOCATION "/Volumes/tri/tri/tri_files/OUTPUT/products_delta"
""")

above it will not work beacuse need specific url 

### creating table instance

In [0]:
from delta.tables import DeltaTable
target_table=DeltaTable.forPath(spark, "/Volumes/tri/tri/tri_files/OUTPUT/products_delta")
display(target_table.toDF())

In [0]:
display(target_table.history())

In [0]:
target_table.delete("ProductID = 'P001'")


In [0]:
display(dbutils.fs.ls("/Volumes/tri/tri/tri_files/OUTPUT/products_delta"))

In [0]:
target_table.delete("ProductID = 'P004'")
display(target_table.toDF())

### 1. Check Current Deletion Vector Status on table


In [0]:
%sql
SHOW TBLPROPERTIES product_catalog
('delta.enableDeletionVectors');

In [0]:
%sql
DESCRIBE DETAIL product_catalog;

### PySpark
### Python

In [0]:
spark.sql("""
SHOW TBLPROPERTIES product_catalog
('delta.enableDeletionVectors')
""").show(truncate=False)

In [0]:
spark.sql("""
DESCRIBE DETAIL product_catalog
""").show(truncate=False)

### 2. Enable Deletion Vectors on table

### SQL

In [0]:
%sql
ALTER TABLE product_catalog
SET TBLPROPERTIES (
  'delta.enableDeletionVectors' = 'true'
);

### PySpark


In [0]:
spark.sql("""
ALTER TABLE product_catalog
SET TBLPROPERTIES (
  'delta.enableDeletionVectors' = 'true'
)
""")

### 3. Disable Deletion Vectors on table

### SQL

In [0]:
%sql
ALTER TABLE product_catalog
SET TBLPROPERTIES (
  'delta.enableDeletionVectors' = 'false'
);

### PySpark

In [0]:
spark.sql("""
ALTER TABLE product_catalog
SET TBLPROPERTIES (
  'delta.enableDeletionVectors' = 'false'
)
""")

### 4. Check All Table Properties

### SQL

In [0]:
%sql
SHOW TBLPROPERTIES product_catalog;

### pyspark

In [0]:
spark.sql("""
SHOW TBLPROPERTIES product_catalog
""").show(truncate=False)

### Now enable the deletion vectores and repeate the same process

### enable the deletion vector on path level

In [0]:
spark.sql("""
ALTER TABLE delta.`/Volumes/tri/tri/tri_files/OUTPUT/products_delta`
SET TBLPROPERTIES (
  'delta.enableDeletionVectors'='True'
)
""")